# Lecture 7: Statistical Modelling Part II

Building on [Lecture 6](lecture_6.ipynb), we move from *fitting a model* to *making it good*: measuring accuracy along multiple axes, controlling complexity, selecting features, tuning hyperparameters, and encoding domain knowledge. We close with tabular foundation models — a genuinely different paradigm that skips gradient descent altogether. [Lecture 8](lecture_8.ipynb) will build on this with causal inference and explainability.

:::{admonition} Learning Objectives
:class: tip
After this lecture, you will be able to:
- Evaluate model accuracy beyond simple metrics (bias, deviance, normalized Gini / CAP)
- Apply regularisation (L1, L2, Elastic Net) to control model complexity {cite}`hoerl1970ridge,tibshirani1996regression`
- Perform feature selection (filter, wrapper, embedded)
- Tune hyperparameters systematically (grid, random, Bayesian)
- Use interaction / monotonicity constraints and custom losses in modern GBMs {cite}`friedman2001greedy,chen2016xgboost,ke2017lightgbm`
- Explain how tabular foundation models use in-context learning {cite}`hollmann2023tabpfn,ye2025tabicl` and when to prefer them over GBDTs {cite}`grinsztajn2022tabular,borisov2022deep`
:::

:::{note}
**Signal forward.** D300 will formalise the theory behind LASSO (oracle properties, sparsity, post-selection inference) and cover **Double Machine Learning** for causal estimation with high-dimensional nuisance parameters. Treat this lecture as the applied ML view; D300 gives the statistical guarantees.
:::

## Table of Contents

- [Model Accuracy](#model-accuracy)
- [Regularisation](#regularisation)
- [Feature Selection](#feature-selection)
- [Hyperparameter Tuning](#hyperparameter-tuning)
- [Advanced Modelling Topics](#advanced-modelling-topics)
- [Tabular Foundation Models](#tabular-foundation-models)
- [Exercises](#exercises)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fun_ds.data import load_california_housing
from fun_ds.plotting import set_lecture_style

set_lecture_style()
rng = np.random.default_rng(42)

# California housing — median house value in $100k units
df = load_california_housing()
target = "MedHouseVal"
feature_names = [c for c in df.columns if c != target]
X = df[feature_names]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Standardised copies for the linear models
scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=feature_names, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=feature_names, index=X_test.index)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Model Accuracy
Multiple dimensions of model quality:

- **Bias**: Are predicted means correct? (compare weighted mean predictions with weighted mean outcomes)
- **Validation error**: Deviance, MSE, MAE, MSPE
- **Ranking of predictions**: How well does the model sort observations? (Normalized Gini / CAP curve)

### Deviance

A generalisation of squared residuals for maximum likelihood estimation (GLMs):

$$D(y, \hat{\mu}) = 2\left(\log p(y|\hat{\theta}_s) - \log p(y|\hat{\theta}_0)\right)$$

where $\hat{\theta}_s$ are the parameters of the saturated (perfect) model and $\hat{\theta}_0$ those of the fitted model. Intuitively, deviance measures how much likelihood we lose by using the fitted model instead of the perfect one. It reduces to the residual sum of squares for Gaussian noise and to the log-loss for Bernoulli outcomes — so a single scoring rule works across GLMs. The lower the deviance, the better.

### Normalized Gini (CAP Curve)

Measures how well the model ranks observations. Adapted from the Lorenz curve in economics:

$$\text{Gini}_{\text{ML}} = \frac{A}{A + B}$$

where the Cumulative Accuracy Profile (CAP) shows what fraction of the total target is captured by the top-ranked predictions.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score

from fun_ds.plotting import plot_residuals

models = {
    "Lasso": Lasso(alpha=0.01, max_iter=10_000),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
}

# Lasso uses scaled inputs; tree models are scale-invariant
inputs_train = {"Lasso": X_train_s, "RandomForest": X_train, "GradientBoosting": X_train}
inputs_test = {"Lasso": X_test_s, "RandomForest": X_test, "GradientBoosting": X_test}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, (name, model) in enumerate(models.items()):
    model.fit(inputs_train[name], y_train)
    y_pred = model.predict(inputs_test[name])

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    bias = float(np.mean(y_pred) - np.mean(y_test))
    print(f"{name:17s}  RMSE = {rmse:.3f}   R2 = {r2:.3f}   bias = {bias:+.4f}")

    # Predicted vs actual
    ax_pa = axes[0, i]
    ax_pa.scatter(y_test, y_pred, alpha=0.3, s=8)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax_pa.plot(lims, lims, "r--", lw=1)
    ax_pa.set_xlabel("Actual")
    ax_pa.set_ylabel("Predicted")
    ax_pa.set_title(f"{name}: predicted vs actual")

    # Residuals
    plot_residuals(y_test, y_pred, ax=axes[1, i], title=f"{name}: residuals")

plt.tight_layout()
plt.show()

## Regularisation

Regularisation prevents overfitting by penalising model complexity. The empirical risk becomes

$$\hat{\beta} = \arg\min_\beta \; \mathcal{L}(y, X\beta) + \Omega(\beta).$$

| Method | Penalty $\Omega(\beta)$ | Reference | Effect |
|--------|-------------------------|-----------|--------|
| Ridge (L2) | $\lambda \|\beta\|_2^2 = \lambda \sum_j \beta_j^2$ | {cite}`hoerl1970ridge` | Shrinks all coefficients smoothly |
| Lasso (L1) | $\lambda \|\beta\|_1 = \lambda \sum_j |\beta_j|$ | {cite}`tibshirani1996regression` | Drives some coefficients to exactly zero |
| Elastic Net | $\lambda_1 \|\beta\|_1 + \lambda_2 \|\beta\|_2^2$ | | Combines sparsity with correlated-feature stability |

The L1 penalty produces sparse solutions because the constraint region has corners at the axes; the L2 penalty has a smooth ball and only shrinks. Elastic Net mixes both, which helps when features are correlated (Lasso alone tends to arbitrarily pick one and ignore the rest).

:::{admonition} Bias-variance role of the penalty
:class: important
Regularisation is a **deliberate injection of bias to buy a larger reduction in variance** — the classical bias-variance trade-off in action {cite}`hastie2009elements`. For Ridge regression, this can be made precise. With SVD $X = U D V^\top$ where $D = \mathrm{diag}(d_1, \dots, d_p)$, the Ridge fit is

$$\hat y_{\mathrm{Ridge}} = X \hat\beta_\lambda = U \, \mathrm{diag}\!\left(\frac{d_j^2}{d_j^2 + \lambda}\right) U^\top y,$$

so each principal component of $X$ is shrunk by the factor $d_j^2 / (d_j^2 + \lambda) \in (0, 1]$. Components with small singular values $d_j$ — the noisy, poorly-identified directions — are shrunk aggressively; components with large $d_j$ are barely touched. The **effective degrees of freedom** of Ridge regression are

$$\mathrm{df}(\lambda) = \mathrm{tr}\!\left[X(X^\top X + \lambda I)^{-1} X^\top\right] = \sum_{j=1}^{p} \frac{d_j^2}{d_j^2 + \lambda},$$

which decreases smoothly from $p$ at $\lambda = 0$ (unpenalised OLS) to $0$ as $\lambda \to \infty$. Because $\mathrm{df}(\lambda) < p$, the estimator is biased, but its variance $\sigma^2 \cdot \mathrm{df}(\lambda) / n$ shrinks with $\lambda$. Optimal $\lambda$ balances the two — see {cite:t}`hastie2009elements` §3.4 for the derivation and the analogous statements for Lasso and Elastic Net.
:::

:::{admonition} Solving Lasso: coordinate descent
:class: note
Lasso has no closed-form solution because the $\ell_1$ penalty is non-differentiable at zero, but its objective is separable and convex. **Coordinate descent** exploits this: cyclically minimise the objective in each coordinate $\beta_j$ while holding the others fixed. For standardised features, the one-dimensional subproblem admits a closed-form soft-thresholding update

$$\beta_j \leftarrow \mathcal{S}_{\lambda}\!\left(\frac{1}{n} \sum_{i=1}^n x_{ij}\bigl(y_i - \hat y_i^{(-j)}\bigr)\right), \quad
\mathcal{S}_{\lambda}(z) = \operatorname{sign}(z)\,\bigl(|z| - \lambda\bigr)_+,$$

where $\hat y_i^{(-j)}$ is the fitted value excluding the $j$-th coordinate. The full "glmnet" algorithm of {cite:t}`friedman2010regularization` combines this with **warm starts along the regularisation path** ($\lambda_{\max} \to \lambda_{\min}$) and **active-set screening** so that only currently-nonzero coordinates are updated — giving the near-linear scaling in $n$ and $p$ that made Lasso practical for genomics and text-scale problems. `sklearn.linear_model.Lasso` and `LassoCV` are direct descendants of this algorithm.
:::

In [ ]:
from sklearn.linear_model import Ridge, Lasso

alphas = np.logspace(-3, 3, 60)

coefs_ridge = np.vstack([
    Ridge(alpha=a).fit(X_train_s, y_train).coef_ for a in alphas
])
coefs_lasso = np.vstack([
    Lasso(alpha=a, max_iter=20_000).fit(X_train_s, y_train).coef_ for a in alphas
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for j, name in enumerate(feature_names):
    ax1.plot(alphas, coefs_ridge[:, j], label=name)
    ax2.plot(alphas, coefs_lasso[:, j], label=name)

for ax, title in [(ax1, "Ridge coefficient path"), (ax2, "Lasso coefficient path")]:
    ax.set_xscale("log")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xlabel(r"$\alpha$ (regularisation strength)")
    ax.set_title(title)
ax1.set_ylabel("Standardised coefficient")
ax2.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.tight_layout()
plt.show()

print("Note how Lasso coefficients hit exactly zero as alpha grows, "
      "while Ridge shrinks them smoothly toward zero.")

## Feature Selection
Methods to reduce the number of features:

1. **Filter methods**: Correlation, mutual information, variance threshold
2. **Wrapper methods**: Forward/backward selection, recursive feature elimination
3. **Embedded methods**: L1 regularisation (built into the model)

:::{note}
Feature selection is less critical for tree-based models which handle irrelevant features naturally, but it improves interpretability and can reduce overfitting for linear models.
:::

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression, RFECV
from sklearn.linear_model import Ridge

# Filter: mutual information (nonparametric, catches nonlinear dependence)
mi_selector = SelectKBest(mutual_info_regression, k=5).fit(X_train, y_train)
mi_scores = pd.Series(mi_selector.scores_, index=feature_names).sort_values(ascending=False)
print("Mutual information with target:")
print(mi_scores.round(3))
print("Selected top-5:", list(mi_scores.head(5).index))

# Wrapper: recursive feature elimination with cross-validation
rfecv = RFECV(
    estimator=Ridge(alpha=1.0),
    step=1,
    cv=5,
    scoring="r2",
    min_features_to_select=1,
    n_jobs=-1,
)
rfecv.fit(X_train_s, y_train)
print(f"\nRFECV optimal number of features: {rfecv.n_features_}")
kept = [f for f, k in zip(feature_names, rfecv.support_) if k]
print("RFECV kept:", kept)

## Hyperparameter Tuning
Hyperparameters control model complexity and must be set before training.

| Strategy | Pros | Cons |
|----------|------|------|
| Grid Search | Exhaustive | Exponential cost |
| Random Search | Efficient in high dimensions | Less systematic |
| Bayesian (Optuna) | Sample-efficient | More complex setup |

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"alpha": np.logspace(-3, 3, 13)}

grid_search = GridSearchCV(
    Ridge(),
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_search.fit(X_train_s, y_train)

print(f"Best alpha       : {grid_search.best_params_['alpha']:.4f}")
print(f"Best CV R^2      : {grid_search.best_score_:.4f}")
print(f"Test R^2 (best)  : {grid_search.score(X_test_s, y_test):.4f}")

# Peek at the CV curve
cv_results = pd.DataFrame(grid_search.cv_results_)
plt.figure(figsize=(8, 4))
plt.errorbar(param_grid["alpha"], cv_results["mean_test_score"],
             yerr=cv_results["std_test_score"], marker="o")
plt.xscale("log")
plt.xlabel(r"$\alpha$")
plt.ylabel("Mean CV R²")
plt.title("Ridge hyperparameter search")
plt.tight_layout()
plt.show()

## Advanced Modelling Topics

Modern gradient boosting frameworks — XGBoost {cite}`chen2016xgboost` and LightGBM {cite}`ke2017lightgbm`, both descendants of Friedman's original gradient boosting machine {cite}`friedman2001greedy` — expose knobs that let us bake domain knowledge directly into the model.

### Interaction constraints

We can restrict which features may interact in the same tree. This encodes domain knowledge (e.g. geographic features should not interact with temporal ones) and often improves generalisation on structured data.

### Monotonicity constraints

Force the model to respect known monotonic relationships:
- Higher income $\rightarrow$ higher house value
- More experience $\rightarrow$ higher salary

Supported natively in LightGBM and XGBoost via `monotone_constraints`.

:::{admonition} Formal statement
:class: note
Modern histogram-based boosters — `sklearn.ensemble.HistGradientBoostingRegressor`, LightGBM, and XGBoost {cite}`chen2016xgboost` — accept a per-feature monotonicity vector $c \in \{-1, 0, +1\}^p$:

- $c_j = +1$ enforces $\hat f(x)$ non-decreasing in $x_j$: $x_j \le x_j' \Rightarrow \hat f(x) \le \hat f(x')$ (all other features fixed),
- $c_j = -1$ enforces non-increasing,
- $c_j = 0$ leaves the feature unconstrained.

Internally, the booster **prunes candidate splits whose child-node predictions would violate the ordering** implied by $c_j$ — the constraint is baked into the tree-growing step rather than added as a penalty after the fact. This is invaluable when domain knowledge or regulation demands specific directionality. Two canonical use cases:

- **Housing:** predicted price should be non-decreasing in square footage — otherwise the model can produce economically absurd point predictions on out-of-sample rows, even if aggregate metrics look fine.
- **Insurance:** predicted premium should be non-decreasing in claim history — a hard requirement in most jurisdictions.

Enforcing monotonicity typically costs a small amount of validation loss (the constrained model has strictly smaller hypothesis space) but eliminates a whole class of failure modes at deployment.
:::

### Custom loss functions in LightGBM

When built-in loss functions don't match your objective, implement your own:
- Provide gradient and hessian functions
- Examples: asymmetric loss, business-specific objectives

In [ ]:
try:
    import lightgbm as lgb

    def asymmetric_mse(y_true, y_pred):
        """Custom loss: penalise under-prediction 2x more than over-prediction.

        Returns (gradient, hessian) with respect to y_pred, as required
        by LightGBM's custom-objective API.
        """
        residual = y_true - y_pred  # positive residual => under-prediction
        grad = np.where(residual > 0, -2.0 * residual, -residual)
        hess = np.where(residual > 0, 2.0, 1.0)
        return grad, hess

    model = lgb.LGBMRegressor(
        objective=asymmetric_mse,
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    over = np.mean(y_pred > y_test)
    under = np.mean(y_pred < y_test)
    print(f"Fraction of predictions ABOVE actual : {over:.3f}")
    print(f"Fraction of predictions BELOW actual : {under:.3f}")
    print("With the asymmetric loss, the model deliberately over-predicts "
          "to avoid the more heavily penalised under-predictions.")

except ImportError:
    print("lightgbm not installed. Install with: pip install lightgbm")
    print("The custom-objective API expects a function returning (grad, hess) "
          "of the loss w.r.t. y_pred; LightGBM will use them to grow trees.")

## Tabular Foundation Models

### From fitting to in-context learning

Every model we have seen so far — from Ridge to LightGBM — is **fit** to the training data via gradient descent or tree-growing. A **tabular foundation model** skips that step entirely.

:::{admonition} What TabPFN actually is
:class: important
{cite:t}`hollmann2023tabpfn` introduced TabPFN as a **Transformer pre-trained on millions of synthetic tabular datasets** drawn from a prior over structural causal models. At inference time, the training rows $(X_{\text{train}}, y_{\text{train}})$ and the query rows $X_{\text{test}}$ are packed into a single sequence and passed through one forward pass of the Transformer; no gradient descent is performed on the user's data. Under mild assumptions on the prior, this forward pass **approximates Bayesian inference** — the network's output distribution converges to the posterior predictive $p(y_{\text{test}} \mid X_{\text{test}}, X_{\text{train}}, y_{\text{train}})$ under the pre-training prior. TabPFN v2 {cite}`hollmann2025tabpfnv2` extends the mechanism to regression and larger datasets, and TabICL {cite}`ye2025tabicl` scales it further via retrieval-augmented in-context learning.
:::

The core idea is that a pre-trained Transformer has already seen enough variation of "tabular supervised learning" that it can generalise to a new dataset **without any gradient updates**. The training data is passed as *context* at prediction time — the same mechanism as few-shot prompting in language models, which you will encounter in D200's Generative AI lecture.

:::{note}
This is called **in-context learning (ICL)**. The model does not change its weights; it uses the labelled training rows as context to predict new rows. `clf.fit(X_train, y_train)` merely stores the training data; `clf.predict(X_test)` runs a single forward pass through the Transformer.
:::

TabPFN v2 supports both **classification and regression** and is available via the `tabpfn` package with a scikit-learn-compatible API.

---

### How does it compare to GBDTs?

Two lines of evidence frame the question. {cite:t}`grinsztajn2022tabular` and the {cite:t}`borisov2022deep` survey established that **well-tuned GBDTs still dominate deep-learning models on typical tabular benchmarks**, especially at scale — an empirical claim that has held up across multiple independent evaluations. More recent independent benchmarks of TabPFN v2 / TabICL {cite}`ye2025tabicl,hollmann2025tabpfnv2` show a nuanced picture:

| Setting | Winner |
|---|---|
| Tiny-to-medium IID datasets (≤ ~10 000 rows) | Tabular foundation models (TabPFN v2 / TabICL competitive with tuned GBDTs) |
| Large datasets (> 50k rows) | GBDTs (XGBoost, LightGBM, CatBoost) {cite}`grinsztajn2022tabular` |
| High-dimensional data (> 100 features) | GBDTs |
| Temporal / grouped (non-IID) splits | GBDTs |

:::{admonition} When NOT to use TabPFN
:class: warning
The original TabPFN v1 {cite}`hollmann2023tabpfn` is deliberately trained on small datasets and inherits three hard practical limits:

1. **Dataset size $n > 10\,000$ rows.** Inference cost is quadratic in $n$ (self-attention over the training rows as context), and the network's inductive bias was tuned for small $n$. TabPFN v2 {cite}`hollmann2025tabpfnv2` relaxes this considerably and TabICL {cite}`ye2025tabicl` pushes further via retrieval, but small-$n$ is still the regime where these models shine.
2. **Feature count $p > 100$.** The tokenisation scheme allocates a fixed input budget per feature; going beyond ~100 features degrades quality sharply.
3. **Heterogeneous categorical dominance.** Datasets whose signal lives mostly in high-cardinality categoricals (e.g. thousands of user IDs, product SKUs, ZIP codes) sit far outside the pre-training prior — GBDTs with target encoding are usually a better fit.

**Empirically, {cite:t}`grinsztajn2022tabular` remains the reference benchmark for the claim that GBDTs still dominate supervised tabular learning at scale**, and any decision to prefer a tabular foundation model on a production problem should be backed by a direct head-to-head evaluation.
:::

Practical takeaway:
- On a new, medium-sized, well-shuffled tabular dataset, **try TabPFN v2 as a strong zero-hyperparameter baseline** — it often matches tuned GBDTs with no tuning effort.
- On large datasets, time-ordered data, or grouped data, **prefer a well-tuned GBDT**.
- The conceptual value is independent of which wins on any given benchmark: in-context learning is a genuinely different computational paradigm.

:::{seealso}
**Forward signals.** The theoretical machinery behind Lasso (oracle properties, sparsity, post-selection inference) is developed in **D300**, and the same course covers **Double Machine Learning** {cite}`chernozhukov2018double` for causal estimation with high-dimensional nuisance parameters — the tool you would reach for if the aim were to *estimate a treatment effect* rather than *predict an outcome* on top of the same regularised nuisance models used here.
:::

---

### Hands-on demo

We compare LogisticRegression, GradientBoosting, and TabPFN on a small subsample of California Housing turned into a binary classification problem (high-value vs low-value house). Small $n$ is where TabPFN shines and where it fits in memory comfortably.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Turn regression into a binary problem: high-value vs low-value
threshold = y_train.median()
y_train_bin = (y_train > threshold).astype(int)
y_test_bin = (y_test > threshold).astype(int)

# Subsample: TabPFN is designed for small n (<= ~10k rows)
sub_idx = rng.choice(len(X_train), size=1000, replace=False)
Xtr_small = X_train_s.iloc[sub_idx]
ytr_small = y_train_bin.iloc[sub_idx]

baselines = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
}

for name, clf in baselines.items():
    clf.fit(Xtr_small, ytr_small)
    proba = clf.predict_proba(X_test_s)[:, 1]
    print(f"{name:20s}  AUC = {roc_auc_score(y_test_bin, proba):.4f}")

# TabPFN — same sklearn API, no gradient descent, no hyperparameters
try:
    from tabpfn import TabPFNClassifier

    clf = TabPFNClassifier()
    clf.fit(Xtr_small.values, ytr_small.values)
    proba = clf.predict_proba(X_test_s.values)[:, 1]
    print(f"{'TabPFN':20s}  AUC = {roc_auc_score(y_test_bin, proba):.4f}")
except ImportError:
    print("TabPFN not installed. Install with: pip install tabpfn")
    print("API is identical to sklearn: clf.fit(X, y); clf.predict_proba(X_test).")
except Exception as exc:  # e.g. model download failure, memory limit
    print(f"TabPFN available but did not run: {type(exc).__name__}: {exc}")

## Exercises

:::{admonition} Exercise 7.1 — Lasso vs Ridge coefficient paths
:class: exercise
Using California Housing, plot Ridge and Lasso coefficient paths side-by-side for $\alpha \in [10^{-3}, 10^{3}]$.
1. At what approximate $\alpha$ does Lasso drive its first coefficient to exactly zero?
2. Which two features survive longest under Lasso? Interpret this in terms of the mutual-information ranking from Section 3.
3. Contrast the shape of Ridge shrinkage (smooth) with Lasso selection (piecewise linear with kinks).
:::

:::{admonition} Exercise 7.2 — RandomizedSearchCV over XGBoost
:class: exercise
Install `xgboost` and tune an `XGBRegressor` on California Housing using `RandomizedSearchCV` with 30 draws over:
- `n_estimators` in `[100, 200, 400, 800]`
- `max_depth` in `[3, 4, 6, 8]`
- `learning_rate` log-uniform in $[10^{-3}, 10^{-1}]$
- `subsample`, `colsample_bytree` uniform in $[0.6, 1.0]$

Report the best CV R², the corresponding hyperparameters, and the test-set R². Compare with the un-tuned `GradientBoostingRegressor` from Section 1.
:::

:::{admonition} Exercise 7.3 — TabPFN vs LogisticRegression vs XGBoost on a small classification problem
:class: exercise
Load `sklearn.datasets.load_breast_cancer` (569 samples, 30 features — squarely in the small-$n$ regime where tabular foundation models are expected to shine). Compare three models via 5-fold cross-validated ROC-AUC:
1. `LogisticRegression` inside a `StandardScaler` pipeline
2. `XGBClassifier` with default hyperparameters
3. `TabPFNClassifier` (wrap the import in `try/except ImportError`)

Which model wins? Does the ranking change if you shrink the training set to 100 rows? Discuss in one paragraph why the *dataset size* matters here, connecting to the "beyond IID" findings above.
:::

:::{admonition} Key Takeaways
:class: important
- Evaluate models on multiple dimensions: bias, validation error, and ranking
- Regularisation (L1/L2) controls complexity — tune the strength via CV
- Feature selection improves interpretability and can reduce overfitting
- Use random search or Bayesian optimisation over grid search for efficiency
- Modern GBMs support constraints (monotonicity, interactions) and custom losses
:::